# DriftWatch customer churn inference

Run from the repository root or `steps/inference`. This notebook downloads the specified model and preprocessing artifacts from S3, validates their compatibility, and exposes one `predict_churn(customers, bundle)` function for a customer dictionary, a list of customer dictionaries, or a pandas DataFrame. Results always contain a `predictions` list in input order.

Use a Python environment with boto3, NumPy, pandas, scikit-learn, and XGBoost **1.7.6**, matching the training requirement. AWS credentials need read access to the three configured S3 objects. No training, registration, or endpoint deployment occurs. Artifact downloads happen only in the explicit loading cell below.

Only load trusted model artifacts: the verified `preprocessing.py` is imported as Python code. Its hashes must match `training_metadata.json`; these hashes establish consistency with the model metadata, not the identity of the publisher. Models without preprocessing provenance must be regenerated and retrained with the current artifact contract.

The prediction and artifact-loading functions are imported from `inference.py` so the notebook and command-line interface share one implementation.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "inference" and PROJECT_ROOT.parent.name == "steps":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from steps.inference.inference import (
    MODEL_S3_URI,
    PREPROCESSING_JSON_S3_URI,
    PREPROCESSING_PY_S3_URI,
)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "inference"


## Download the model and matching preprocessing

Each download uses a fresh directory so files from previous runs cannot be mixed in. Only the expected regular files at the archive root are copied out; archive paths and links are never extracted. Bundled preprocessing takes precedence, and the separate S3 files must match it byte for byte. If the mutable preprocessing S3 paths have been overwritten since training, use the original matching objects instead.

In [2]:
from steps.inference.inference import download_artifacts, unpack_model


In [3]:
from steps.inference.inference import validate_artifacts, load_inference_bundle


## Shared prediction function

Pass raw feature values using IBM or canonical column names, without a churn label. The saved preprocessing implementation handles aliases and feature ordering; extra columns are ignored according to its contract. Missing required fields, invalid numeric values, and unknown or missing categories reject the entire request. No partial batch results are returned.

A dictionary means one customer; a list of dictionaries or DataFrame means a batch. Empty requests are rejected. Results preserve row order (DataFrame index labels are not returned). The saved threshold is applied with `probability >= threshold`.

In [4]:
from steps.inference.inference import predict_churn


## Load your S3 artifacts

Executing the next cell downloads the three configured S3 objects into the ignored `artifacts/inference/` directory and loads the verified model. Downloads fail rather than use unverified preprocessing. To reuse a previously downloaded directory without AWS access, call `load_inference_bundle(path_to_directory)` instead.

In [5]:
model_directory = download_artifacts(
    MODEL_S3_URI, PREPROCESSING_JSON_S3_URI, PREPROCESSING_PY_S3_URI, ARTIFACT_DIR
)
bundle = load_inference_bundle(model_directory)
print("Loaded model:", model_directory)
print("Feature count:", bundle["metadata"]["feature_count"])
print("Selected threshold:", bundle["metadata"]["selected_threshold"])
print("Required raw features:", bundle["state"]["required_features"])
print("Allowed categories:", bundle["state"]["categories"])

Loaded model: /home/sagemaker-user/DriftWatch/artifacts/inference/model-af223db84da24cff8003771a74088ec1
Feature count: 30
Selected threshold: 0.55
Required raw features: ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Allowed categories: {'Gender': ['Female', 'Male'], 'SeniorCitizen': ['No', 'Yes'], 'Partner': ['No', 'Yes'], 'Dependents': ['No', 'Yes'], 'PhoneService': ['No', 'Yes'], 'MultipleLines': ['No', 'No phone service', 'Yes'], 'InternetService': ['DSL', 'Fiber optic', 'No'], 'OnlineSecurity': ['No', 'No internet service', 'Yes'], 'OnlineBackup': ['No', 'No internet service', 'Yes'], 'DeviceProtection': ['No', 'No internet service', 'Yes'], 'TechSupport': ['No', 'No internet service', 'Yes'], 'StreamingTV': ['No', 'No internet se

## Single-customer and batch examples

The following synthetic customer is generated from the saved schema so every required field is present. Categorical values use recorded reference levels and numeric fields use illustrative values. Replace these with actual customer records; this is a usage demonstration, not model-quality validation. All three calls below use the same function and should agree.

In [6]:
example_customer = {}
for name in bundle["state"]["required_features"]:
    if bundle["state"]["feature_types"][name] == "categorical":
        example_customer[name] = bundle["state"]["reference_levels"][name]
    else:
        example_customer[name] = {"tenure": 12, "MonthlyCharges": 70.0, "TotalCharges": 840.0}.get(name, 0)

single_result = predict_churn(example_customer, bundle)
batch_result = predict_churn([example_customer, example_customer.copy()], bundle)
dataframe_result = predict_churn(pd.DataFrame([example_customer, example_customer]), bundle)
assert batch_result["predictions"] == single_result["predictions"] * 2
assert dataframe_result == batch_result
print("Single customer:", json.dumps(single_result, indent=2))
print("Batch:", json.dumps(batch_result, indent=2))

Single customer: {
  "predictions": [
    {
      "churn_probability": 0.4450671672821045,
      "churn_prediction": 0
    }
  ]
}
Batch: {
  "predictions": [
    {
      "churn_probability": 0.4450671672821045,
      "churn_prediction": 0
    },
    {
      "churn_probability": 0.4450671672821045,
      "churn_prediction": 0
    }
  ]
}


For a file of raw, unlabeled customer records, use `predict_churn(pd.read_csv("customers.csv"), bundle)`. Do not pass the headerless, already-transformed training/test CSVs: this function expects raw named features and applies preprocessing itself.

This notebook is a local inference implementation. Endpoint handlers and model registration/deployment wiring are separate work; `steps/inference/inference.py` provides the shared functions and a JSON/CSV command-line interface.